# Ablation — UNM lambda/schedule study


# UNM Lambda Ablation (corrected schedule)

**Purpose**: decide final `lambda_u` (0.05 vs 0.1) and confirm that the corrected schedule (`semi_start_epoch=15`, `patience_es=40`) sufficiently reduces pre-SSL checkpoints.

> **These runs are NOT final paper results.** They exist only for decision-making. The final rerun will live in `runs_final_v1/` with clean experiment names once the ablation decision is made.

**Output root**: `runs_lambda_ablation/` (separate from historical `runs/`)

**Visualization**: automatic visualization is **disabled** (`save_preds_vis=False`). Use the post-hoc visualization cell at the end of this notebook to generate figures for any selected run.

**Skip logic** (per cell):
- **Case A (skip)**: `best_model.pt` + `test_metrics.csv` exist -> skip training & eval
- **Case B (eval only)**: `best_model.pt` exists but `test_metrics.csv` or `*_run_report.json` missing -> skip training, re-run eval only
- **Case C (train)**: `best_model.pt` missing -> full training

**Execution order**: seed_0 (9 experiments) -> seed_1 (9) -> seed_2 (9). Total: 27 experiment cells.

> **IMPORTANT LEGACY WARNING**: `01_train_eval_colab.ipynb` contains 42 legacy cells (RUN 97-123) from a previous iteration pointing to `runs/` as output. **Do NOT execute those legacy cells** - they mix ablation outputs with historical runs. The canonical ablation lives HERE in `05_unm_lambda_ablation.ipynb`.


In [ ]:
# ============================================================
# SETUP — run this cell once after every runtime restart
# Do NOT run training here. Select one experiment cell below.
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

!git clone https://github.com/sebastianquispearias/tesis-seg.git
%cd tesis-seg
!pip install -q -r requirements.txt

# --- Environment fingerprint (for reproducibility debugging) ---
import torch, sys
print("Python    :", sys.version)
print("torch     :", torch.__version__)
print("CUDA      :", torch.version.cuda)
!git log --oneline -1
!pip show albumentations | grep Version
!nvidia-smi | grep -E "NVIDIA|Driver Version|CUDA Version"
# --------------------------------------------------------------

import sys
sys.path.append("/content/tesis-seg")

from src.defaults import get_default_config, summarize_config
from src.augmentations import (
    get_supervised_train_augmentation,
    get_weak_augmentation,
    get_strong_augmentation,
)
from src.datasets import (
    build_supervised_datasets,
    build_unlabeled_datasets,
    build_dataloaders,
)
from src.train import run_training
from src.evaluate import evaluate_checkpoint
from src.visualization import show_dataset_examples, show_predictions

# ── Debug fingerprint helpers ─────────────────────────────────
import json, os, platform, subprocess, time, importlib.metadata

def _fp_get_version(pkg):
    try: return importlib.metadata.version(pkg)
    except Exception: return None

def _fp_git_hash_from_src(src_train_file):
    # Derive repo root from src/train.py: {repo_root}/src/train.py
    repo_root = os.path.dirname(os.path.dirname(os.path.abspath(src_train_file)))
    try:
        return subprocess.check_output(
            ["git", "log", "--oneline", "-1"], cwd=repo_root, stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception: return None

def _fp_save(pre, post, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump({"pre_run": pre, "post_run": post}, f, indent=2, default=str)

def _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                    unlabeled_ds=None, temporal_unlab_ds=None):
    import torch, sys
    from src.models import create_model

    # 1. Environment
    try: gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "n/a"
    except: gpu_name = "n/a"
    env = {
        "python":      sys.version,
        "torch":       torch.__version__,
        "torchvision": _fp_get_version("torchvision"),
        "cuda":        torch.version.cuda,
        "cudnn":       str(torch.backends.cudnn.version()) if torch.cuda.is_available() else "n/a",
        "segmentation_models_pytorch": _fp_get_version("segmentation-models-pytorch"),
        "albumentations": _fp_get_version("albumentations"),
        "platform":    platform.platform(),
        "gpu_name":    gpu_name,
    }

    # 2. Code provenance — git hash derived from actual runtime src path
    import src.train, src.datasets, src.defaults, src.evaluate
    provenance = {
        "src_train":    src.train.__file__,
        "src_datasets": src.datasets.__file__,
        "src_defaults": src.defaults.__file__,
        "src_evaluate": src.evaluate.__file__,
        "git_hash":     _fp_git_hash_from_src(src.train.__file__),
    }

    # 3. Effective config
    cfg_keys = [
        "seed", "arch", "backbone", "n_classes",
        "image_preproc", "mask_smoothing", "target_size", "use_pad", "imagenet_norm",
        "batch_size", "num_workers", "drop_last", "num_augmented",
        "lr", "weight_decay", "epochs", "warmup_epochs", "patience_es", "eval_threshold",
        "use_semi", "use_temp_consistency",
        "lambda_u", "tau", "ema_decay", "semi_start_epoch", "semi_warmup_epochs", "lambda_t",
        "unlabeled_subdir", "exp_dir",
    ]
    eff_cfg = {k: cfg.get(k) for k in cfg_keys}
    unlab_loader = loaders.get("unlabeled_loader")
    eff_cfg["batch_size_unlab"] = unlab_loader.batch_size if unlab_loader is not None else None

    # 4. Dataset / loader facts
    ds_facts = {
        "len_train_ds":          len(train_ds),
        "len_val_ds":            len(val_ds),
        "len_test_ds":           len(test_ds),
        "len_unlabeled_ds":      len(unlabeled_ds) if unlabeled_ds is not None else None,
        "len_temporal_unlab_ds": len(temporal_unlab_ds) if temporal_unlab_ds is not None else None,
        "train_loader_batch_size":      loaders["train_loader"].batch_size,
        "train_loader_num_workers":     loaders["train_loader"].num_workers,
        "train_loader_drop_last":       loaders["train_loader"].drop_last,
        "unlabeled_loader_batch_size":  unlab_loader.batch_size if unlab_loader else None,
        "unlabeled_loader_num_workers": unlab_loader.num_workers if unlab_loader else None,
        "unlabeled_loader_drop_last":   unlab_loader.drop_last if unlab_loader else None,
    }

    # 5. Sample identifiers — reads .files attribute, no IO beyond what dataset already did
    try: sup_ids = train_ds.files[:5]
    except Exception as e: sup_ids = f"unavailable: {e}"
    try: unl_ids = unlabeled_ds.files[:5] if unlabeled_ds is not None else None
    except Exception as e: unl_ids = f"unavailable: {e}"
    sample_ids = {"first5_train": sup_ids, "first5_unlabeled": unl_ids}

    # 6. Batch tensor shapes — analytical, no DataLoader consumed, no RNG touched
    try:
        H, W = cfg["target_size"]
        C = 3  # IMREAD_COLOR: grayscale PNGs expand to 3 identical channels
        eff_bs = cfg["batch_size"] * (1 + cfg.get("num_augmented", 0))  # flatten_collate
        bs_u = max(1, cfg["batch_size"] // 4)  # mirrors datasets.py build_dataloaders
        batch_shapes = {
            "xb":   [eff_bs, C, H, W],
            "yb":   [eff_bs, 1, H, W],
            "xw_u": [bs_u, C, H, W] if unlab_loader is not None else None,
            "xs_u": [bs_u, C, H, W] if unlab_loader is not None else None,
            "note": "analytically derived from cfg — no DataLoader consumed",
        }
    except Exception as e:
        batch_shapes = {"error": str(e)}

    # 7. Model fingerprint — RNG save/restore so training is unaffected.
    # Belt-and-suspenders: run_training() also calls seed_everything(seed) first.
    try:
        import random as _random, numpy as _np
        _rng = {
            "py":   _random.getstate(),
            "np":   _np.random.get_state(),
            "th":   torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        }
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        model_fp = {
            "total_params":          sum(p.numel() for p in _m.parameters()),
            "trainable_params":      sum(p.numel() for p in _m.parameters() if p.requires_grad),
            "first_state_dict_keys": list(_m.state_dict().keys())[:8],
        }
        del _m
        _random.setstate(_rng["py"])
        _np.random.set_state(_rng["np"])
        torch.set_rng_state(_rng["th"])
        if _rng["cuda"] is not None:
            torch.cuda.set_rng_state_all(_rng["cuda"])
    except Exception as e:
        model_fp = {"error": str(e)}

    return {
        "timestamp_utc":     time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "environment":       env,
        "provenance":        provenance,
        "effective_cfg":     eff_cfg,
        "dataset_facts":     ds_facts,
        "sample_ids":        sample_ids,
        "batch_shapes":      batch_shapes,
        "model_fingerprint": model_fp,
    }


def _fp_collect_post(artifacts, results):
    history = artifacts.get("history") or []
    best_row = max(history, key=lambda r: r.get("val_iou_global", 0.0)) if history else None
    vm = (results or {}).get("val_metrics", {})
    tm = (results or {}).get("test_metrics", {})
    return {
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "best_path":     artifacts.get("best_path"),
        "best_epoch_info": {
            "epoch":          best_row.get("epoch") if best_row else None,
            "val_iou_global": best_row.get("val_iou_global") if best_row else None,
            "val_loss":       best_row.get("val_loss") if best_row else None,
        },
        "val_metrics": {
            "f1_global":       vm.get("global_f1"),
            "iou_global":      vm.get("global_iou"),
            "f1_sample_mean":  vm.get("sample_mean_f1"),
            "iou_sample_mean": vm.get("sample_mean_iou"),
        },
        "test_metrics": {
            "f1_global":       tm.get("global_f1"),
            "iou_global":      tm.get("global_iou"),
            "f1_sample_mean":  tm.get("sample_mean_f1"),
            "iou_sample_mean": tm.get("sample_mean_iou"),
        },
        "finished_successfully": True,
        "exception": None,
    }
# ──────────────────────────────────────────────────────────────

print("Imports OK — select one experiment cell below and run it.")

## Seed 0 — all 9 UNM experiments


In [ ]:
# === RUN 1/27: supervised_p40/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_p40"
_SEED     = 0
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only (ablation: patience_es=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 2/27: semi_all_lateral_lam005_s15p40/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_all_lateral_lam005_s15p40"
_SEED     = 0
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (ablation: corrected schedule + lambda)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM corrected schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 3/27: semi_all_lateral_lam010_s15p40/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_all_lateral_lam010_s15p40"
_SEED     = 0
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (ablation: corrected schedule + lambda)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.1
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM corrected schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 4/27: mean_teacher_all_lateral_lam005_s15p40/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_all_lateral_lam005_s15p40"
_SEED     = 0
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (ablation: corrected schedule + lambda)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM corrected schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 5/27: mean_teacher_all_lateral_lam010_s15p40/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_all_lateral_lam010_s15p40"
_SEED     = 0
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (ablation: corrected schedule + lambda)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.1
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM corrected schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 6/27: semi_r10_lam005_s15p40/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r10_lam005_s15p40"
_SEED     = 0
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (ablation: corrected schedule + lambda)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM corrected schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 7/27: semi_r10_lam010_s15p40/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r10_lam010_s15p40"
_SEED     = 0
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (ablation: corrected schedule + lambda)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.1
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM corrected schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 8/27: mean_teacher_r10_lam005_s15p40/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r10_lam005_s15p40"
_SEED     = 0
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (ablation: corrected schedule + lambda)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM corrected schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 9/27: mean_teacher_r10_lam010_s15p40/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r10_lam010_s15p40"
_SEED     = 0
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (ablation: corrected schedule + lambda)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.1
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM corrected schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Seed 1 — all 9 UNM experiments


In [ ]:
# === RUN 10/27: supervised_p40/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_p40"
_SEED     = 1
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only (ablation: patience_es=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 11/27: semi_all_lateral_lam005_s15p40/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_all_lateral_lam005_s15p40"
_SEED     = 1
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (ablation: corrected schedule + lambda)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM corrected schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 12/27: semi_all_lateral_lam010_s15p40/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_all_lateral_lam010_s15p40"
_SEED     = 1
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (ablation: corrected schedule + lambda)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.1
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM corrected schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 13/27: mean_teacher_all_lateral_lam005_s15p40/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_all_lateral_lam005_s15p40"
_SEED     = 1
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (ablation: corrected schedule + lambda)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM corrected schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 14/27: mean_teacher_all_lateral_lam010_s15p40/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_all_lateral_lam010_s15p40"
_SEED     = 1
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (ablation: corrected schedule + lambda)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.1
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM corrected schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 15/27: semi_r10_lam005_s15p40/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r10_lam005_s15p40"
_SEED     = 1
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (ablation: corrected schedule + lambda)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM corrected schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 16/27: semi_r10_lam010_s15p40/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r10_lam010_s15p40"
_SEED     = 1
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (ablation: corrected schedule + lambda)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.1
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM corrected schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 17/27: mean_teacher_r10_lam005_s15p40/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r10_lam005_s15p40"
_SEED     = 1
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (ablation: corrected schedule + lambda)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM corrected schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 18/27: mean_teacher_r10_lam010_s15p40/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r10_lam010_s15p40"
_SEED     = 1
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (ablation: corrected schedule + lambda)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.1
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM corrected schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Seed 2 — all 9 UNM experiments


In [ ]:
# === RUN 19/27: supervised_p40/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "supervised_p40"
_SEED     = 2
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only (ablation: patience_es=40)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 20/27: semi_all_lateral_lam005_s15p40/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_all_lateral_lam005_s15p40"
_SEED     = 2
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (ablation: corrected schedule + lambda)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM corrected schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 21/27: semi_all_lateral_lam010_s15p40/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_all_lateral_lam010_s15p40"
_SEED     = 2
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (ablation: corrected schedule + lambda)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.1
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM corrected schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 22/27: mean_teacher_all_lateral_lam005_s15p40/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_all_lateral_lam005_s15p40"
_SEED     = 2
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (ablation: corrected schedule + lambda)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM corrected schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 23/27: mean_teacher_all_lateral_lam010_s15p40/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_all_lateral_lam010_s15p40"
_SEED     = 2
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (ablation: corrected schedule + lambda)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.1
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM corrected schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_all_lateral/images"

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 24/27: semi_r10_lam005_s15p40/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r10_lam005_s15p40"
_SEED     = 2
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (ablation: corrected schedule + lambda)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM corrected schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 25/27: semi_r10_lam010_s15p40/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "semi_r10_lam010_s15p40"
_SEED     = 2
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (ablation: corrected schedule + lambda)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "pseudo_label"
cfg["lambda_u"]             = 0.1
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM corrected schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 26/27: mean_teacher_r10_lam005_s15p40/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r10_lam005_s15p40"
_SEED     = 2
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (ablation: corrected schedule + lambda)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM corrected schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 27/27: mean_teacher_r10_lam010_s15p40/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r10_lam010_s15p40"
_SEED     = 2
# Ablation output root (separate from historical runs/)
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation/{_EXP_NAME}/seed_{_SEED}"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (ablation: corrected schedule + lambda)
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.1
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15   # UNM corrected schedule
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (ablation: patience_es=40)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40   # corrected schedule
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Ablation: disable automatic visualization
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

# --- Robust 3-case skip-logic ---
_exp_dir = cfg["exp_dir"]
_best_path_skip = os.path.join(_exp_dir, "best_model.pt")
_metrics_path_skip = os.path.join(_exp_dir, "test_metrics.csv")
import glob as _glob_skip
_has_best = os.path.isfile(_best_path_skip)
_has_metrics = os.path.isfile(_metrics_path_skip)
_has_report = len(_glob_skip.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
_skip_all = _has_best and _has_metrics
_eval_only = _has_best and (not _has_metrics or not _has_report)

if _skip_all:
    print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado (Case A)")
elif _eval_only:
    print(f"Partial run detected for {_EXP_NAME}/seed_{_SEED}: best_model exists but evaluation/report files are missing (Case B)")

if not _skip_all:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    if cfg.get("use_semi", False):
        weak_tf   = get_weak_augmentation(cfg)
        strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    if cfg.get("use_semi", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
            cfg, weak_tf=weak_tf, strong_tf=strong_tf
        )
    else:
        unlabeled_ds, temporal_unlab_ds = None, None

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        if _eval_only:
            # Case B: re-evaluate only, no training
            from src.models import create_model
            _model_reeval = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _model_reeval, loaders, _best_path_skip, [])
            artifacts = {"model": _model_reeval, "best_path": _best_path_skip, "history": []}
        else:
            # Case C: full training
            artifacts = run_training(cfg, loaders)
            results = evaluate_checkpoint(
                cfg,
                artifacts['model'],
                loaders,
                artifacts['best_path'],
                artifacts['history'],
            )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Post-hoc visualization

Edit `EXP_NAME` and `SEED` below to visualize any completed run.


In [ ]:
# === POST-HOC VISUALIZATION ===
# Edit RUN_ROOT, EXP_NAME, SEED below and re-run this cell to generate
# prediction figures for an existing trained run. Does NOT retrain.

RUN_ROOT = "/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_lambda_ablation"
EXP_NAME = "semi_all_lateral_lam005_s15p40"   # <-- EDIT
SEED = 0                                       # <-- EDIT

import os, json, torch
from src.models import create_model
from src.augmentations import get_supervised_train_augmentation
from src.datasets import build_supervised_datasets, build_dataloaders
from src.evaluate import evaluate_checkpoint

_exp_dir = f"{RUN_ROOT}/{EXP_NAME}/seed_{SEED}"
_cfg_path = os.path.join(_exp_dir, "config.json")
with open(_cfg_path) as f:
    cfg = json.load(f)
# Force viz ON only for this post-hoc pass
cfg["save_preds_vis"] = True
cfg["exp_dir"] = _exp_dir

train_tf = get_supervised_train_augmentation(cfg)
_, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
loaders = build_dataloaders(cfg, train_ds=None, val_ds=val_ds, test_ds=test_ds,
                             unlabeled_ds=None, temporal_unlab_ds=None)

model = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
_best_path = os.path.join(_exp_dir, "best_model.pt")
results = evaluate_checkpoint(cfg, model, loaders, _best_path, [])
print(f"Visualization saved to: {os.path.join(_exp_dir, 'preds_vis')}")
